# Qdrant RAG Integration

Wintermute supports **Qdrant** as a vector store backend for RAG knowledge bases.
Qdrant is a high-performance vector database that can run locally (on-disk) or as a
remote server, making it ideal for persistent, scalable knowledge bases.

This notebook covers:
1. Configuration variants (remote URL vs local `db_path`)
2. Creating a local Qdrant collection programmatically
3. Embedding and upserting documents
4. How `bootstrap_rags()` discovers and registers Qdrant KBs
5. Querying Qdrant-backed RAG providers
6. Metadata fields (`document_types`, `created_at`)

## Prerequisites

Install the required packages (already included in Wintermute's dependencies):

```bash
pip install qdrant-client llama-index-vector-stores-qdrant sentence-transformers
```

For a remote Qdrant server, start one with Docker:

```bash
docker run -p 6333:6333 qdrant/qdrant
```

For local on-disk usage, no server is needed — Qdrant runs embedded in-process.

## Configuration Variants

Qdrant KBs are configured via `rag_config.json` with `"vector_store_type": "qdrant"`.
There are two connection modes:

In [ ]:
import json

# Variant 1: Remote Qdrant server
remote_config = {
    "rag_id": "red_team_manuals_v1",
    "description": "Embedded systems and red team exploit manuals.",
    "base_provider_id": "bedrock",
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "vector_store_type": "qdrant",
    "qdrant_url": "http://localhost:6333",
    "qdrant_collection_name": "red_team_manuals",
    "document_types": ["pdf", "markdown"],
    "created_at": "2026-03-01T12:00:00Z",
}

# Variant 2: Local on-disk Qdrant (no server needed)
local_config = {
    "rag_id": "firmware_docs",
    "description": "Firmware analysis documentation.",
    "base_provider_id": "bedrock",
    "embedding_model": "BAAI/bge-small-en-v1.5",
    "vector_store_type": "qdrant",
    "db_path": "./storage/firmware_docs",
    "qdrant_collection_name": "firmware_docs",
    "document_types": ["pdf", "text"],
    "created_at": "2026-03-01T12:00:00Z",
}

print("Remote Qdrant config:")
print(json.dumps(remote_config, indent=2))
print()
print("Local Qdrant config:")
print(json.dumps(local_config, indent=2))

## Local Qdrant

When using `db_path`, Qdrant runs as an embedded database — no server process needed.
The data is persisted to disk at the specified path.

If neither `qdrant_url` nor `db_path` is specified, Wintermute auto-creates a
`qdrant_db/` directory inside the knowledge base folder itself.

Priority order:
1. `qdrant_url` — connect to remote server
2. `db_path` — use local on-disk database at specified path
3. Neither — auto-create `<kb_folder>/qdrant_db/`

In [ ]:
# Create a local Qdrant client and collection programmatically.
# This demonstrates how Qdrant works at the client level.

from qdrant_client import QdrantClient, models

# Create an in-memory client for demonstration (use path= for persistence)
client = QdrantClient(location=":memory:")

# Create a collection with vector dimensions matching bge-small-en-v1.5 (384-dim)
client.create_collection(
    "demo_kb",
    vectors_config=models.VectorParams(
        size=384,
        distance=models.Distance.COSINE,
    ),
)

print(f"Collections: {client.get_collections()}")

In [ ]:
# Embed documents with HuggingFace and upsert into Qdrant.

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

texts = [
    "JTAG pinout: TCK=pin3, TMS=pin5, TDI=pin7, TDO=pin9, TRST=pin11",
    "VCC_CORE operates at 1.1V with a tolerance of +/- 5%",
    "SPI flash can be read using flashrom with the ch341a programmer",
    "UART baud rate is 115200, 8N1, accessible on header J4",
]

embeddings = model.encode(texts).tolist()
print(f"Embedded {len(texts)} documents, vector dim: {len(embeddings[0])}")

# Upsert into Qdrant
client.upsert(
    collection_name="demo_kb",
    points=[
        models.PointStruct(id=i, vector=emb, payload={"text": text})
        for i, (emb, text) in enumerate(zip(embeddings, texts))
    ],
)

print(f"Upserted {len(texts)} points into 'demo_kb'")

In [ ]:
# Query the collection directly to verify embeddings work.

query = "What voltage does the core use?"
query_vector = model.encode(query).tolist()

results = client.query_points(
    collection_name="demo_kb",
    query=query_vector,
    limit=2,
)

print(f"Query: {query}")
print()
for point in results.points:
    print(f"  Score: {point.score:.4f} | {point.payload['text']}")

## Bootstrapping Qdrant KBs

When `bootstrap_rags()` runs, it scans `knowledge_bases/` and `external_repos/` for
subdirectories. For each directory:

1. Parse `rag_config.json` to determine `vector_store_type`
2. If `"qdrant"`, build a `QdrantVectorStore` using the configured URL or path
3. Create a `RAGProvider` with the Qdrant vector store
4. Register as `rag-<folder_name>` in the LLM registry

No `storage_db/` directory is required for Qdrant KBs.

In [ ]:
# Educational: bootstrap flow for Qdrant KBs.
# bootstrap_rags() performs these steps automatically.
#
# from wintermute.ai.bootstrap import bootstrap_rags
#
# bootstrap_rags scans knowledge_bases/ for rag_config.json files.
# When it finds vector_store_type="qdrant", it:
#   1. Creates a QdrantClient (remote or local)
#   2. Wraps it in a QdrantVectorStore
#   3. Passes the vector_store to RAGProvider.__init__()
#   4. RAGProvider creates a VectorStoreIndex.from_vector_store()
#   5. Registers the provider as rag-<name>

print("bootstrap_rags(registry) discovers Qdrant KBs automatically.")
print()
print("Discovery logic:")
print("  1. Has storage_db/ dir?  -> local file-based KB")
print("  2. Has vector_store_type='qdrant'?  -> Qdrant-backed KB")
print("  3. Neither?  -> skipped")

## Querying Qdrant-backed RAG Providers

Once bootstrapped, Qdrant-backed providers work identically to local file-based ones.
Use `router.set_default(provider="rag-<name>")` to activate them.

In [ ]:
# Educational: querying a Qdrant-backed RAG provider.
# This follows the same pattern as any other RAG provider.

# from wintermute.ai.bootstrap import init_router
# from wintermute.ai.types import ChatRequest, Message
#
# router = init_router()
# router.set_default(provider="rag-red_team_manuals_v1")
#
# req = ChatRequest(messages=[Message(role="user", content="How to dump firmware via JTAG?")])
# provider, routed_req = router.choose(req)
# response = provider.chat(routed_req)
# print(response.content)

print("Querying a Qdrant-backed RAG provider:")
print("  router.set_default(provider='rag-<name>')")
print("  provider, req = router.choose(chat_request)")
print("  response = provider.chat(req)")
print()
print("The RAG provider transparently:")
print("  1. Queries the Qdrant vector store for relevant chunks")
print("  2. Augments the prompt with retrieved context")
print("  3. Forwards to the base LLM (e.g., Bedrock)")

## Metadata Fields: `document_types` and `created_at`

These fields are informational metadata for knowledge base management:

- **`document_types`**: Lists the types of documents in the KB (e.g., `["pdf", "markdown"]`).
  Useful for filtering and inventory.
- **`created_at`**: ISO 8601 timestamp for when the KB was created or last re-indexed.
  Useful for tracking freshness.

In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

config_path = Path("knowledge_bases/qdrant_example/rag_config.json")
if config_path.exists():
    config = json.loads(config_path.read_text())
    created = datetime.fromisoformat(config["created_at"])
    age_days = (datetime.now(timezone.utc) - created).days
    print(f"KB: {config.get('rag_id', 'unknown')}")
    print(f"  Description: {config.get('description', '')}")
    print(f"  Document types: {config.get('document_types', [])}")
    print(f"  Created: {config['created_at']}")
    print(f"  Age: {age_days} days")
else:
    print("Run from the examples/ directory to see the qdrant_example config")

## Summary

Qdrant integration adds a scalable, persistent vector database backend to Wintermute's RAG system.

**Local vs Remote Qdrant:**
- **Local** (`db_path`): No server needed, data persisted to disk, ideal for development
- **Remote** (`qdrant_url`): Shared server, scalable, ideal for production and team use

**Config reference:**
- `vector_store_type: "qdrant"` — enables Qdrant backend
- `qdrant_url` — remote server URL (takes priority over `db_path`)
- `db_path` — local on-disk database path
- `qdrant_collection_name` — collection name (defaults to folder name)

**Tips:**
- Use `QDRANT_API_KEY` env var for authenticated remote servers
- The `BAAI/bge-small-en-v1.5` model produces 384-dim vectors — set collection accordingly
- Qdrant KBs don't need `storage_db/` — the vector store is the database itself
- All RAG querying patterns (`router.set_default()`, `ai rag use`) work identically